# Sports Activity

Author: Taran Schlichtmann

Course: GB885 Python Fundamentals

---

## Business Case
Jon Paul Sports Management Group (JPSMG) is a sports talent agency that represents
athletes with the support of machine learning and predictive modeling. Its business
model depends on aggressive data collection from a variety of sources to supplement
the firm's internal data store.

Acting as an analyst for JPSMG, this notebook ingests historic data requested by four
internal teams, loads each source into a pandas DataFrame, and answers the questions
each team needs for decision-making.

---

## Data Sources

| Team | Source | Ingestion method |
|---|---|---|
| Olympic Account | female_olympic_swimmers.csv | Local file upload |
| Strategy | Ali-Ce Athletes.csv | Public GitHub raw URL |
| Basketball Account | nbaapi.com player totals | Live REST API |
| Social Media | ESPN 2017 World Fame 100 | Kaggle API |

---
# Olympic Account Team
The Olympic Account team acquired a dataset of female Olympic swimmers to enhance
its modeling and athlete representation. The file is downloaded from a
local file system, then uploaded into Colab and read into a DataFrame.

In [91]:
# Import library needed to load csv into dataframe
import pandas as pd       # data loading, cleaning, and analysis

In [92]:
# Upload female_olympic_swimmers.csv from the local file system, then load it
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]       # grab the uploaded files name
fos_df = pd.read_csv(filename)


fos_df.head()     # preview column names, dtypes, and formatting

Saving female_olympic_swimmers.csv to female_olympic_swimmers (1).csv


,Unnamed: 0,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,474,259,Reema Abdo,F,21.0,173.0,59.0,Canada,CAN,1984 Summer,1984,Summer,Los Angeles,Swimming,Swimming Women's 100 metres Backstroke,NaN
1,475,259,Reema Abdo,F,21.0,173.0,59.0,Canada,CAN,1984 Summer,1984,Summer,Los Angeles,Swimming,Swimming Women's 200 metres Backstroke,NaN
2,476,259,Reema Abdo,F,21.0,173.0,59.0,Canada,CAN,1984 Summer,1984,Summer,Los Angeles,Swimming,Swimming Women's 4 x 100 metres Medley Relay,Bronze
3,517,290,Fatima Abdul Majeed Hameed Al-Kirashi,F,14.0,NaN,NaN,Bahrain,BRN,2000 Summer,2000,Summer,Sydney,Swimming,Swimming Women's 50 metres Freestyle,NaN
4,729,417,Sara Helena berg,F,17.0,190.0,73.0,Sweden,SWE,1988 Summer,1988,Summer,Seoul,Swimming,Swimming Women's 50 metres Freestyle,NaN


## Data Inspection
Inspect the raw file for the common issues such as size, data types, missing values,
duplicates, category counts, and outliers before answering any questions.

In [93]:
# Dataset size (rows, columns)
fos_df.shape

(9850, 16)

In [94]:
# Column names, data types, and non-null counts
fos_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9850 entries, 0 to 9849
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  9850 non-null   int64  
 1   ID          9850 non-null   int64  
 2   Name        9850 non-null   object 
 3   Sex         9850 non-null   object 
 4   Age         9761 non-null   float64
 5   Height      8572 non-null   float64
 6   Weight      8463 non-null   float64
 7   Team        9850 non-null   object 
 8   NOC         9850 non-null   object 
 9   Games       9850 non-null   object 
 10  Year        9850 non-null   int64  
 11  Season      9850 non-null   object 
 12  City        9850 non-null   object 
 13  Sport       9850 non-null   object 
 14  Event       9850 non-null   object 
 15  Medal       1374 non-null   object 
dtypes: float64(3), int64(3), object(10)
memory usage: 1.2+ MB


In [95]:
# Missing values per column
# Age, Height, and Weight carry nulls; left as-is for now since dropping them would
# discard otherwise-usable athlete records. Revisiting only if required.
fos_df.isnull().sum()

,0
Unnamed: 0,0
ID,0
Name,0
Sex,0
Age,89
Height,1278
Weight,1387
Team,0
NOC,0
Games,0


In [96]:
# Full-row duplicates
print("Exact duplicate rows:", fos_df.duplicated().sum())

Exact duplicate rows: 0


In [97]:
# Summary statistics as quick scan for outliers / impossible values in the numeric fields
fos_df.describe()

,Unnamed: 0,ID,Age,Height,Weight,Year
count,9850.000000,9850.000000,9761.000000,8572.000000,8463.000000,9850.000000
mean,137777.659594,69358.410660,19.487450,171.468735,61.482748,1987.533807
std,78048.336179,38905.628561,3.774357,7.067459,6.585295,22.472969
min,474.000000,259.000000,11.000000,131.000000,39.000000,1912.000000
25%,70271.250000,35773.000000,17.000000,167.000000,57.000000,1972.000000
50%,140348.500000,70454.000000,19.000000,171.000000,61.000000,1992.000000
75%,204017.250000,102427.250000,22.000000,176.000000,66.000000,2004.000000
max,270943.000000,135489.000000,41.000000,193.000000,85.000000,2016.000000


## Data Cleaning
Drop the unnamed index column carried over from the CSV, and label
missing medals explicitly. Most Olympians never medal, so a null Medal is meaningful as
it means "did not medal," not "data missing."

In [98]:
# Drop the leftover unnamed index column (errors='ignore' makes this safe to re-run)
fos_df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

# Label non-medaling athletes explicitly so 'Medal' has no nulls
fos_df['Medal'] = fos_df['Medal'].fillna('No Medal')

fos_df.isnull().sum()   # confirm Medal nulls are resolved

/tmp/ipykernel_3318/3867638205.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  fos_df['Medal'].fillna('No Medal', inplace=True)


,0
ID,0
Name,0
Sex,0
Age,89
Height,1278
Weight,1387
Team,0
NOC,0
Games,0
Year,0


## Q1 — Which country produced the most female Olympic swimmers (2000–2016)?
Filter to the 2000–2016 games, then de-duplicate by athlete so a swimmer who
competed in multiple events or years is counted only once per country.

In [99]:
# Filter to the 2000-2016 games
filtered_df = fos_df[(fos_df['Year'] >= 2000) & (fos_df['Year'] <= 2016)]

# Count each athlete once per country (a swimmer enters many events / multiple games)
unique_swimmers = filtered_df.drop_duplicates(subset=['ID', 'NOC'])

# Rank countries by number of unique female swimmers produced
top_countries = unique_swimmers['NOC'].value_counts()
print(top_countries.head(10))


NOC
USA    83
CHN    78
AUS    69
CAN    56
GBR    51
GER    50
JPN    48
RUS    46
FRA    42
ITA    42
Name: count, dtype: int64


**Finding:** The USA produced the most unique female Olympic swimmers from
2000–2016, ahead of Australia and China.

## Q2 — Tallest female medalist of the 2016 Games
A swimwear company launching a line for taller women wants an endorsement candidate,
limited to 2016 competitors. Filter to 2016 medalists and select the maximum height.

In [100]:
# Restrict to 2016, then to athletes who won a medal
filtered_df = fos_df[fos_df['Year'] == 2016]
medalists = filtered_df[filtered_df['Medal'] != 'No Medal']

# Select the single tallest medalist (idxmax ignores NaN heights automatically)
tallest_medalist_2016 = medalists.loc[medalists['Height'].idxmax()]
print(tallest_medalist_2016)

ID                                                 103490
Name                                         Cierra Runge
Sex                                                     F
Age                                                  20.0
Height                                              193.0
Weight                                               85.0
Team                                        United States
NOC                                                   USA
Games                                         2016 Summer
Year                                                 2016
Season                                             Summer
City                                       Rio de Janeiro
Sport                                            Swimming
Event     Swimming Women's 4 x 200 metres Freestyle Relay
Medal                                                Gold
Name: 7460, dtype: object


**Finding:** Cierra Runge (USA) is the tallest 2016 female medalist at 193 cm,
a gold medalist in the 4×200 m freestyle relay and would be the recommended endorsement candidate.

---
# Strategy Team
The Strategy Team studies trends among the world's highest-paid athletes to guide
recruitment of high-value clients. The dataset is loaded directly from a public GitHub
repository via its raw URL.

(This data is from 2014; per the assignment, treated as current.)

In [101]:
# Load Athletes.csv straight from the raw GitHub URL
url = 'https://raw.githubusercontent.com/ali-ce/datasets/refs/heads/master/Most-paid-athletes/Athletes.csv'
most_paid_athletes_df = pd.read_csv(url)

most_paid_athletes_df.head()

,Rank,Name,Sport,Total Pay,Salary/Winnings,Endorsements,Nation,Gender,Year of birth,Birth Date,Place of Birth,Height (cm),Wikipedia Page,dbpedia Page,Image,Description
0,55,Aaron Rodgers,Football,"$22,000,000","$14,500,000","$7,500,000",United States,Male,1983,2/12/1983,"Chico, California, United States",188,http://en.wikipedia.org/wiki/Aaron_Rodgers,dbpedia.org/resource/Aaron_Rodgers,http://commons.wikimedia.org/wiki/Special:File...,"Aaron Charles Rodgers (born December 2, 1983) ..."
1,95,Adam Scott,Golf,"$17,700,000","$8,700,000","$9,000,000",Australia,Male,1980,16/07/1980,"Adelaide, Australia",183,https://en.wikipedia.org/wiki/Adam_Scott_(golfer),dbpedia.org/resource/Adam_Scott_(golfer),http://commons.wikimedia.org/wiki/Special:File...,Adam Derek Scott (born 16 July 1980) is an Aus...
2,60,Adrian Gonzalez,Baseball,"$21,500,000","$21,100,000","$400,000",United States,Male,1982,8/05/1982,"San Diego, California, United States",188,http://en.wikipedia.org/wiki/Adrian_Gonzalez,dbpedia.org/resource/Adrian_Gonzalez,http://commons.wikimedia.org/wiki/Special:File...,"Adrian Gonzalez (born May 8, 1982), also known..."
3,48,Alex Rodriguez,Baseball,"$22,900,000","$22,600,000","$300,000",United States,Male,1975,27/07/1975,New York City,190,http://en.wikipedia.org/wiki/Alex_Rodriguez,dbpedia.org/resource/Alex_Rodriguez,http://commons.wikimedia.org/wiki/Special:File...,"Alexander Emmanuel "" Alex "" Rodriguez (born Ju..."
4,93,Alfonso Soriano,Baseball,"$18,050,000","$18,000,000","$50,000",Dominican Republic,Male,1976,7/01/1976,"San Pedro de Macorís, Dominican Republic",185,http://en.wikipedia.org/wiki/Alfonso_Soriano,dbpedia.org/resource/Alfonso_Soriano,http://commons.wikimedia.org/wiki/Special:File...,"Alfonso Guilleard Soriano (born January 7, 197..."


## Clean Total Pay Column
Total Pay reads in as text because of the leading $ and comma separators. Strip both
characters and cast to a numeric type once, here, so every downstream calculation
works on real numbers.

In [102]:
# Adjusting Total Pay to remove the characters '$' and ','
most_paid_athletes_df['Total Pay'] = most_paid_athletes_df['Total Pay'].str.replace('$', '')

most_paid_athletes_df['Total Pay'] = most_paid_athletes_df['Total Pay'].str.replace(',', '')

## Q3 & Q4 — Correlation between height and total pay

In [103]:
# Correlation between athlete height and total pay, rounded to two decimals
round(most_paid_athletes_df['Total Pay'].corr(most_paid_athletes_df['Height (cm)']), 2)

np.float64(-0.11)

**Finding (Q3 & Q4):** The correlation is -0.11 which represents a weak, negative relationship.
Height has essentially no meaningful bearing on how much a top athlete earns.

## Q5 — Age spread of top earners (interquartile range)
Top athletes have a narrow prime-earning window. Measure it with the interquartile
range (75th − 25th percentile) of ages. Because the year offset is constant, the IQR is
identical no matter which "current" year we assume.

In [104]:
# what is the difference between the 75th percentile and 25th percentile of top paid athletes ages
age = 2026 - most_paid_athletes_df['Year of birth']
age.describe()

# 75% - 25% age
round(age.describe()[6] - age.describe()[4], 2)

/tmp/ipykernel_3318/1160975798.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  round(age.describe()[6] - age.describe()[4], 2)
/tmp/ipykernel_3318/1160975798.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  round(age.describe()[6] - age.describe()[4], 2)


np.float64(5.25)

**Finding:** The middle 50% of top-paid athletes fall within a 5.25-year age band showing
a narrow prime-earning window that reinforces the value of identifying talent
early.

## Q6 — Which sport pays the most on average?

In [105]:
# compare the mean total pay by sport
most_paid_athletes_df['Total Pay'] = most_paid_athletes_df['Total Pay'].astype(int)

# convert pay out of scientific notation
pd.set_option('display.float_format', '{:.2f}'.format)
most_paid_athletes_df.groupby('Sport')['Total Pay'].mean()

,Total Pay
Sport,
Baseball,21694444.44
Basketball,29272222.22
Boxing,48950000.00
Cricket,30000000.00
Football,22690294.12
Golf,35520000.00
Racing,25016666.67
Soccer,31500000.00
Tennis,33966666.67


**Finding:** Boxing has the highest average pay (~$48.95M), well ahead of the next
sports — a handful of marquee pay-per-view fighters pull the sport's average up.

---
# Basketball Account Team
The Basketball team uses a live REST API of per-season player statistics to support
summer contract negotiations. We call the 2024 season endpoint and load the response
into a DataFrame.

In [106]:
# install libraries
!pip install requests

In [107]:
# import modules
import json

In [108]:
# API call for the 2024 season
# The API caps each response at 50 players per page, so we page through
# until there are no more results — this assembles the complete league.
base_url = 'https://api.server.nbaapi.com/api/playertotals?season=2024'

all_players = []
page = 1
while True:
    response = requests.get(f'{base_url}&page={page}&pageSize=100')
    page_data = response.json()['data']
    if not page_data:          # empty page means we've reached the end
        break
    all_players.extend(page_data)
    page += 1

player_stats_2024 = pd.DataFrame(all_players)

player_stats_2024.shape

(949, 32)

## Q7 — Most games started, and how many players reached it

In [109]:
# maximum number of games started by a player
player_stats_2024['gamesStarted'].max()

82

**Finding:** The most games started is 82 (a full season), achieved by 6 players.

In [110]:
# how many players started 82 games
player_stats_2024[player_stats_2024['gamesStarted'] == 82]

,playerId,playerName,position,age,games,gamesStarted,minutesPg,fieldGoals,fieldAttempts,fieldPercent,...,totalRb,assists,steals,blocks,turnovers,personalFouls,points,team,season,isPlayoff
22,greenja05,Jalen Green,SG,21,82,82,2601,563,1332,0.42,...,423,291,66,28,191,115,1610,HOU,2024,False
23,bridgmi01,Mikal Bridges,SF,27,82,82,2854,564,1294,0.44,...,372,299,81,30,164,116,1606,BRK,2024,False
25,sabondo01,Domantas Sabonis,C,27,82,82,2928,634,1068,0.59,...,1120,673,74,48,272,250,1593,SAC,2024,False
43,holmgch01,Chet Holmgren,C,21,82,82,2413,505,953,0.53,...,648,200,53,190,131,197,1357,OKC,2024,False
94,valanjo01,Jonas Valančiūnas,C,31,82,82,1925,402,719,0.56,...,721,173,32,68,111,218,1002,NOP,2024,False
95,barneha02,Harrison Barnes,PF,31,82,82,2381,347,732,0.47,...,249,99,54,12,58,98,1000,SAC,2024,False


## Q8 — Underused defensive standouts ("Stocks per Game")
"Stocks" = steals + blocks, a shorthand for defensive disruption. The team wants strong
defenders who are underused at under 25 minutes per game labeling them as undervalued recruiting
targets. Build both per-game features, filter to underused players, and rank by stocks
per game.

In [111]:
# strong defensive players by stocks per game
# 'stocks_per_game' = sum of steals and blocks
# 'minutes_per_game' = avg minutes played in a game (total minutes played (minutesPG) divided by games played)
# looking at players who are underused with less than 25 minutes_per_game
# create a list of the 10 underused players with the highest stocks per game

player_stats_2024['stocks_per_game'] = (player_stats_2024['steals'] + player_stats_2024['blocks']) / player_stats_2024['games']
player_stats_2024['minutes_per_game'] = (player_stats_2024['minutesPg']) / player_stats_2024['games']
underused_players = player_stats_2024[player_stats_2024['minutes_per_game'] < 25]
underused_players.sort_values(by='stocks_per_game', ascending=False).head(10)


,playerId,playerName,position,age,games,gamesStarted,minutesPg,fieldGoals,fieldAttempts,fieldPercent,...,steals,blocks,turnovers,personalFouls,points,team,season,isPlayoff,stocks_per_game,minutes_per_game
140,gaffoda01,Daniel Gafford,PF,25,74,66,1815,348,480,0.72,...,65,153,74,227,814,2TM,2024,False,2.95,24.53
224,kesslwa01,Walker Kessler,C,22,64,22,1493,229,350,0.65,...,30,154,66,132,518,UTA,2024,False,2.88,23.33
837,covinro01,Robert Covington,PF,33,3,3,69,3,9,0.33,...,6,2,0,7,9,LAC,2024,False,2.67,23.00
322,gaffoda01,Daniel Gafford,PF,25,29,21,623,145,186,0.78,...,21,56,28,80,324,DAL,2024,False,2.66,21.48
303,thybuma01,Matisse Thybulle,SF,26,65,19,1487,126,317,0.40,...,113,49,40,93,354,POR,2024,False,2.49,22.88
685,williro04,Robert Williams,C,26,6,0,119,17,26,0.65,...,7,7,7,17,41,POR,2024,False,2.33,19.83
842,batumni01,Nicolas Batum,PF,35,3,0,54,3,8,0.38,...,3,4,1,5,8,LAC,2024,False,2.33,18.00
461,robinmi01,Mitchell Robinson,C,25,31,21,768,73,127,0.57,...,37,35,25,55,173,NYK,2024,False,2.32,24.77
579,porzikr01,Kristaps Porziņģis,C,28,7,4,165,28,60,0.47,...,5,11,5,15,86,BOS,2024,True,2.29,23.57
406,easonta01,Tari Eason,PF,22,22,0,480,88,189,0.47,...,31,19,20,51,215,HOU,2024,False,2.27,21.82


**Finding:** Daniel Gafford leads all underused players in stocks per game.

---
# Social Media Team
The Social Media Team studies athletes' online reach. The source is a Kaggle dataset
(2017 ESPN World Fame 100), retrieved through the Kaggle API and read from Excel.

In [112]:
# install kaggle library
!pip install kaggle


In [113]:
import os      # file-system handling for the Kaggle download

In [114]:
os.makedirs('/content/kaggle_data/', exist_ok=True)

In [116]:
from google.colab import files

# The kaggle.json file needs to be uploaded by the user to authenticate with Kaggle.
# Run this cell, then click 'Choose Files' to upload your kaggle.json.
print("Please upload your kaggle.json file.")
files.upload()

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Move the uploaded kaggle.json to the .kaggle directory
!mv /content/kaggle.json ~/.kaggle/

# Set read-write permissions for the owner only to secure the credentials
!chmod 600 ~/.kaggle/kaggle.json

Please upload your kaggle.json file.


Saving kaggle.json to kaggle.json


In [117]:
# call kaggle data from rishidamarla/2017-espn-athletes-fame-rankings using kaggle api
! kaggle datasets download rishidamarla/2017-espn-athletes-fame-rankings

Dataset URL: https://www.kaggle.com/datasets/rishidamarla/2017-espn-athletes-fame-rankings
License(s): CC0-1.0
2017-espn-athletes-fame-rankings.zip: Skipping, found more recently modified local copy (use --force to force download)


In [118]:
# unzip data file
! unzip 2017-espn-athletes-fame-rankings.zip

Archive:  2017-espn-athletes-fame-rankings.zip
replace 2017 ESPN World Fame 100.xlsx? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [119]:
# save the excel file 2017 ESPN World Fame 100.xlsx to a pandas dataframe

athletes_df = pd.read_excel('2017 ESPN World Fame 100.xlsx')

## Data Inspection

In [120]:
# Identify size of dataset (Rows, Columns)
athletes_df.shape

(100, 15)

In [121]:
# Inspect the first few rows
athletes_df.head()

,Rank,PY Rank,Last Name,First Name,Sport,Country,Team,Endorsements,Instgram Followers,Facebook Followers,Twitter Followers,Endorsements.1,Instgram Followers.1,Facebook Followers.1,Twitter Followers.1
0,1,1,Ronaldo,Cristiano,Soccer,Portugal,Real Madrid,32.00,93.00,118.10,50.40,32000000,93000000.00,118100000,50400000.00
1,2,2,James,LeBron,Basketball,USA,Cleveland Cavaliers,55.00,28.50,22.60,34.40,55000000,28500000.00,22600000,34400000.00
2,3,3,Messi,Lionel,Soccer,Argentina,Barcelona,28.00,65.10,86.60,NaN,28000000,65100000.00,86600000,0.00
3,4,5,Federer,Roger,Tennis,Switzerland,NaN,60.00,2.80,14.20,6.70,60000000,2800000.00,14200000,6700000.00
4,5,13,Mickelson,Phil,Golf,USA,NaN,50.00,NaN,NaN,NaN,50000000,0.00,0,0.00


In [122]:
# Check dtypes and nulls
athletes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Rank                  100 non-null    int64  
 1   PY Rank               100 non-null    object 
 2   Last Name             100 non-null    object 
 3   First Name            100 non-null    object 
 4   Sport                 100 non-null    object 
 5   Country               100 non-null    object 
 6   Team                  63 non-null     object 
 7   Endorsements          96 non-null     float64
 8   Instgram Followers    91 non-null     float64
 9   Facebook Followers    92 non-null     float64
 10  Twitter Followers     92 non-null     float64
 11  Endorsements.1        100 non-null    int64  
 12  Instgram Followers.1  100 non-null    float64
 13  Facebook Followers.1  100 non-null    int64  
 14  Twitter Followers.1   100 non-null    float64
dtypes: float64(6), int64(3),

In [123]:
# null counts per column
athletes_df.isnull().sum()

,0
Rank,0
PY Rank,0
Last Name,0
First Name,0
Sport,0
Country,0
Team,37
Endorsements,4
Instgram Followers,9
Facebook Followers,8


In [124]:
# Check for any duplicates
athletes_df.duplicated().sum()

np.int64(0)

## Q9 — Which platform's following correlates most with endorsements?
Compare endorsements against each platform's follower count using the unscaled columns.

In [125]:
# Correlation of endorsements with each platform's follower count
print("Instagram:", round(athletes_df['Endorsements'].corr(athletes_df['Instgram Followers']), 2))
print("Facebook :", round(athletes_df['Endorsements'].corr(athletes_df['Facebook Followers']), 2))
print("Twitter  :", round(athletes_df['Endorsements'].corr(athletes_df['Twitter Followers']), 2))

Instagram: 0.23
Facebook : 0.23
Twitter  : 0.36


**Finding:** Twitter following has the strongest correlation with endorsements
(0.36, vs. 0.23 for both Instagram and Facebook) which is the most useful single signal of
endorsement potential in this dataset.

## Q10 — Total following and endorsements by sport
Combine the three platforms into a single reach metric, then find the sport that leads in
average total followers and the sport that leads in average endorsements.

In [126]:
# sum of followers between all three social media platforms
athletes_df['total_followers'] = athletes_df[['Instgram Followers', 'Facebook Followers', 'Twitter Followers']].sum(axis=1)

print(athletes_df.groupby('Sport')['total_followers'].mean().idxmax())
print(athletes_df.groupby('Sport')['Endorsements'].mean().idxmax())

Soccer
Track and Field


**Finding:** Soccer has the highest average total following, while Track and Field
has the highest average endorsements reach. Findings signal that endorsement value does not always live in
the same sport, a useful nuance for JPSMG's marketing strategy.

---
# Key Findings for JPSMG

- Olympic Account: The USA produced the most female Olympic swimmers (2000–2016);
  Cierra Runge (193 cm, 2016 gold) is the recommended endorsement fit for a tall-women's
  swimwear line.
- Strategy: Height and pay are essentially unrelated (r = −0.11). Top earners cluster
  in a narrow 5.25-year age band, and **Boxing** pays the most on average — recruit early
  and weight marketability over physical profile.
- Basketball Account: Across the full 2024 season, 6 players started all 82 games, and
  Daniel Gafford stands out as the top underused defensive target (highest stocks per
  game under 25 minutes) for value-driven contract negotiations.
- Social Media: Twitter following tracks endorsements most closely; Soccer leads in
  average reach while Track and Field leads in average endorsements.